# Import 

In [1]:
# ! pip install fast_plate_ocr --break-system-packages


In [2]:
import os
import onnxruntime as ort
import time
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.models import load_model
import tf2onnx
import onnx
import sys
import os 
import torch
import pynvml
import threading

I0000 00:00:1785937962.146420    4618 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# Load prerequisite

In [3]:
import onnxruntime as ort
print(ort.get_available_providers())


['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [4]:
project_dir=os.getcwd()
print(project_dir)


images = "test_images"
# keras_model = "models/best.keras"
keras_model_path = r"/workspace/models/best.keras"
image_folder=os.path.join(os.getcwd(),images)
keras_model_path = os.path.join(os.getcwd(),keras_model_path)

# print("model_path:- ",model_path_onnx)
print("image_folder:-",image_folder)
print("Keras_model_path",keras_model_path)

/workspace
image_folder:- /workspace/test_images
Keras_model_path /workspace/models/best.keras


In [5]:
# Convert 

In [6]:
# if not os.path.exists(keras_model_path):
#     print(f"\nERROR: The file was not found at: {keras_model_path}")
#     print("Please double-check the path you specified above.\n")
#     sys.exit(1)


# try:
#     from fast_plate_ocr.train.model.layers import MaxBlurPooling2D
# except ImportError:
#     print("\nERROR: Could not import 'MaxBlurPooling2D'.")
#     print("Please ensure 'fast_plate_ocr' is installed in your Python environment.")
#     print("Run: pip install fast_plate_ocr\n")
#     sys.exit(1)

# custom_objects_dict = {
#     'MaxBlurPooling2D': MaxBlurPooling2D
# }

# print(f"Loading Keras model from: {keras_model_path}...")
# model = load_model(keras_model_path, custom_objects=custom_objects_dict, compile=False)

# print("Successfully loaded model. Starting ONNX conversion...")

# onnx_model, _ = tf2onnx.convert.from_keras(model)

# output_path = 'keras_model1.onnx'
# onnx.save(onnx_model, output_path)

# print(f"Successfully converted and saved to: {output_path}")

# Load ONNX Model

In [7]:
model_onnx = "models/best.onnx"
model_path_onnx=os.path.join(os.getcwd(),model_onnx)

In [8]:
IMG_HEIGHT = 64
IMG_WIDTH = 128
ALPHABET = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ_"

PAD = "_"

# Preprocess Image

In [9]:
def preprocess_images_onnx(folder_path):
    start_pre = time.perf_counter()
    image_files = sorted(
        [f
        for f in os.listdir(folder_path)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png", ".bmp", ".webp",".avif"))])

    images = []


    for file in image_files:
        path = os.path.join(folder_path, file)
        image = cv2.imread(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(
            image,
            (IMG_WIDTH, IMG_HEIGHT),
            interpolation=cv2.INTER_LINEAR,)

        image = image.astype(np.uint8)

        images.append(image)

    batch = np.stack(images)

    return batch, image_files,(time.perf_counter()-start_pre)*100


# Onnx Session Creation

In [10]:
# session = ort.InferenceSession(
#     model_path_onnx,
#     providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
# )

# print(session.get_providers())


# Inference 

In [11]:

def inference_onnx(onnx_model, batch_numpy, warmup_runs=10):
    ort.preload_dlls()
    opts = ort.SessionOptions()
    opts.add_session_config_entry("session.required_cuda_compute_capability", "0") 
    
    session = ort.InferenceSession(
        str(onnx_model), 
        providers=['CUDAExecutionProvider', 'CPUExecutionProvider'], 
        sess_options=opts
    )

    input_name = session.get_inputs()[0].name
    input_feed = {input_name: batch_numpy}

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    # --- COLD RUN ---
    start_cold = time.perf_counter()
    outputs = session.run(None, input_feed)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    inference_cold_ms = (time.perf_counter() - start_cold) * 1000

    # --- WARMUP RUNS (To stabilize GPU clocks) ---
    if warmup_runs > 0:
        for _ in range(warmup_runs):
            _ = session.run(None, input_feed)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    # --- WARM RUN TIMING INTERMEDIARY (Averaged over 10 iterations) ---
    iterations = 10
    start_warm = time.perf_counter()
    for _ in range(iterations):
        outputs = session.run(None, input_feed)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    # Calculate the average time for a single inference pass across all iterations
    inference_warm_ms = ((time.perf_counter() - start_warm) * 1000) / iterations
    
    return outputs[0], inference_cold_ms, inference_warm_ms



# Decode Output

In [12]:
def decode(outputs, filenames):
    start_post = time.perf_counter()
    results = []

    for file, pred in zip(filenames, outputs):
        indices = np.argmax(pred, axis=1)
        scores = np.max(pred, axis=1)

        plate = []
        confidence = []

        for idx, score in zip(indices, scores):
            ch = ALPHABET[idx]
            if ch == PAD:
                break
            plate.append(ch)
            confidence.append(score)

        plate_str = "".join(plate)
        mean_conf = float(np.mean(confidence)) if confidence else 0.0

        # print("-" * 60)
        # print("Image      :", file)
        # print("Prediction :", plate_str)
        # print("Confidence :", f"{mean_conf:.4f}")

        results.append({"image": file,"plate": plate_str,"confidence": mean_conf})
    end_post = time.perf_counter()
    postprocess_time_ms = (end_post - start_post) * 1000
    return results,postprocess_time_ms



In [13]:
def main(model_path_onnx, image_folder):
    has_nvml = False
    peak_vram_bytes = 0
    base_vram_bytes = 0
    stop_tracking = False

    try:
        pynvml.nvmlInit()
        nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        # Capture baseline VRAM allocation before processing begins
        base_vram_bytes = pynvml.nvmlDeviceGetMemoryInfo(nvml_handle).used
        peak_vram_bytes = base_vram_bytes
        has_nvml = True
    except Exception:
        print("Warning: NVML initialization failed. GPU memory tracking disabled.")

    def track_peak_memory():
        nonlocal peak_vram_bytes
        while not stop_tracking:
            try:
                mem_info = pynvml.nvmlDeviceGetMemoryInfo(nvml_handle)
                if mem_info.used > peak_vram_bytes:
                    peak_vram_bytes = mem_info.used
            except Exception:
                pass
            time.sleep(0.002) # Tighter sleep cycle for higher accurate sampling

    if has_nvml:
        mem_thread = threading.Thread(target=track_peak_memory, daemon=True)
        mem_thread.start()
        
    # Run pipeline
    batch, files, preprocess_ms = preprocess_images_onnx(image_folder)
    predictions, inf_cold_ms, inf_warm_ms = inference_onnx(model_path_onnx, batch, warmup_runs=5)
    results, postprocess_ms = decode(predictions, files)
    print("=" * 60)
    print("Results")
    print("=" * 60)
    for i in range(len(results)):
        print(results[i])
        print(i)
    
    stop_tracking = True
    if has_nvml:
        mem_thread.join()

  
    net_peak_vram_bytes = max(0, peak_vram_bytes - base_vram_bytes)
    peak_mem_mb = net_peak_vram_bytes / (1024 * 1024)
    
    if has_nvml:
        pynvml.nvmlShutdown()  
        
    batch_size = len(files)
    avg_preprocess = preprocess_ms / batch_size
    avg_postprocess = postprocess_ms / batch_size
    
    print("=" * 60)
    print("ONNX RUNTIME PROFILE RUN SUMMARY (FIXED)")
    print("=" * 60)
    print(f"Total Images Processed : {batch_size}")
    print(f"Preprocess Latency     : {avg_preprocess:.2f} ms per image")
    # print(f"Inference (Cold Run)   : {inf_cold_ms / batch_size:.2f} ms per image")
    print(f"Inference (Warm Run)   : {inf_warm_ms / batch_size:.2f} ms per image")
    print(f"Postprocess Latency    : {avg_postprocess:.2f} ms per image")
    print(f"Model VRAM Allocation  : {peak_mem_mb:.2f} MB")


In [14]:
if __name__ == "__main__":
    main(model_path_onnx,image_folder)

Results
{'image': '234.jpeg', 'plate': 'AJ38BN8563', 'confidence': 0.596328854560852}
0
{'image': '81aIaTwEBXL._AC_UF1000,1000_QL80_.jpg', 'plate': 'KA41CR4547', 'confidence': 0.879433810710907}
1
{'image': 'IMG_20240120_151956.jpg', 'plate': 'RJ11DT3249', 'confidence': 0.9397951364517212}
2
{'image': 'Screenshot from 2026-08-05 13-16-41.png', 'plate': 'GJ03ER0563', 'confidence': 0.9881545901298523}
3
{'image': 'Screenshot from 2026-08-05 13-23-22.png', 'plate': 'MH02B14322', 'confidence': 0.8985115885734558}
4
{'image': 'Screenshot from 2026-08-05 13-23-44.png', 'plate': 'TN06BE3335', 'confidence': 0.3692696690559387}
5
{'image': 'Screenshot from 2026-08-05 13-24-03.png', 'plate': 'KA01EC5588', 'confidence': 0.9872317314147949}
6
{'image': 'Untitled.jpeg', 'plate': 'AB01C12294', 'confidence': 0.9036809802055359}
7
{'image': 'images.jpeg', 'plate': 'AP27AL1432', 'confidence': 0.9057513475418091}
8
{'image': 'images1.jpeg', 'plate': 'MA01MW9787', 'confidence': 0.7488687634468079}
9
{'im